Model Created by TAMUCC Beto Estrada

Model Training: trains and evaluates MLP models that predict water level (Port Lavaca, Six Mile Park, ...).

**How to use (the short version)**
1. Write one config file for the experiment (see **Config files** below) and commit it to the repo's `configs/` folder.
2. Runtime → Run all. Sign in to Google if asked. Nothing trains yet.
3. In **Pick experiments**, turn ON the experiments you want.
4. Run **Train selected experiments**.
5. Open the one `<session>_report.html` in the session folder. That's the result.

Data files and configs are downloaded automatically from the GitHub repo by the **Setup** cell below. No uploading needed.

In [ ]:
#@title Run this to download data, configs + code from GitHub. { display-mode: "form" }
REPO_URL = "https://github.com/conrad-blucher-institute/coastal_inundation.git"  #@param {type:"string"}
DRIVE_FOLDER = "wl_experiments"  #@param {type:"string"}

import os, subprocess
from IPython.display import display, HTML

# retrieves the data from the github repo; clones it into the vm
REPO_DIR = "/content/wl-repo"
if not os.path.exists(REPO_DIR):
    result = subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Couldn't download the repo. Check REPO_URL.\n{result.stderr}")
else:
    # Already downloaded this session: pull to pick up config edits made on GitHub
    result = subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Couldn't get the latest version from GitHub, using the copy already here.\n{result.stderr}")

COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()
DATA_DIR = f"{REPO_DIR}/data"
CONFIG_DIR = f"{REPO_DIR}/configs"
print(f"Repo ready (version {COMMIT})")

# Connect Google Drive (if this fails, training still runs and results go to downloads)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
    DRIVE_OK = True
    print(f"Drive connected. Results will be saved to MyDrive/{DRIVE_FOLDER}")
except Exception as e:
    OUT_DIR = "/content/wl_outputs"
    DRIVE_OK = False
    display(HTML("<b style='color:red'>Google Drive did not connect. Training will still run, but the downloaded "
                 "zip files will be the ONLY copy of the results, and the master results CSV will not be updated "
                 f"this session.</b><br><span style='color:gray'>({e})</span>"))

os.makedirs(OUT_DIR, exist_ok=True)

# All the functions live in the repo. -i runs the file as if it were pasted here, so it can use the
# values above and every cell below can use its functions. Run this cell again to get new code.
CODE_FILE = f"{REPO_DIR}/code/wl_functions.py"
if not os.path.exists(CODE_FILE):
    raise FileNotFoundError(f"{CODE_FILE} is not in the repo. Upload code/wl_functions.py to GitHub, then run this cell again.")
%run -i {CODE_FILE}
print("Code loaded")

Data and configs come from the GitHub repo (see the Setup cell above):

- `data/` holds the data CSVs
- `configs/` holds the config files. **One config file = one experiment.**
- `code/wl_functions.py` holds all the functions (data loading, config checks, model, report, training), with comments explaining each part. To change the code, edit that file on GitHub, then run the Setup cell again.

# Config files

A config file is a CSV with the columns `setting,value,notes`. The `notes` column is optional and only there for people to read. **Every setting must be filled in.** If one is missing, the experiment shows up in red in **Pick experiments** with a message saying what to fix, and it can't be run.

| setting | what it means | example |
|---|---|---|
| `location` | name shown in the report | `Port Lavaca` |
| `train_files` | data files for training, separated by `;` | `PortLavaca2022.csv;Rain_2022_Iowia.csv` |
| `val_files` | data files for validation, separated by `;` | `PortLavaca2023.csv;Rain_2023_Iowia.csv` |
| `test_files` | data files for testing, separated by `;`. Never used for training; its results only appear in the separate VAL+TEST report | `PortLavaca2024.csv;Rain_2024_Iowia.csv` |
| `lead_hrs` | lead times to train, separated by `;` | `6;12;18;24;48` |
| `rain` | `off`, `reported`, `to_now` or `perfprog` (see below) | `perfprog` |
| `rain_future_hrs` | perfprog only: hours of future rain (max 18). Use `0` for the other rain settings | `18` |
| `poc` | use Port O'Connor water level: `yes` or `no` | `yes` |
| `train_gap_fill`, `val_gap_fill`, `test_gap_fill` | for each data set: fill water level / wind / POC gaps up to this many hours (`0` = off). Rain is never filled | `1`, `0`, `0` |
| `normalize` | `yes` rescales every input to mean 0 and standard deviation 1, using the training rows only. The scaling is saved inside the model. The target stays in meters. `no` = raw inputs | `yes` |
| `lookback_hrs` | hours of past data the model sees | `24` |
| `layers`, `neurons`, `dropout`, `activation`, `epochs`, `patience` | model settings | `1`, `20`, `0.15`, `relu`, `10000`, `50` |
| `min_delta` | early stopping: the smallest drop in validation loss that counts as an improvement. Smaller gains don't reset `patience`, so training stops sooner | `0.001` |
| `n_models` | how many models to train per lead time. The report shows each one and their average | `3` |
| `seed` | random seed of model 1. Model 2 uses seed+1, and so on | `42` |

The model settings and `lookback_hrs` can be either one value for every lead time, or one value per lead time separated by `;` (e.g. `dropout` = `0.15;0.1;0.1;0.2;0.15`).

**Rain settings**
- `off`: no rain input.
- `reported`: past rain only, stopping 2 h before the forecast is issued (only rain that would already be published).
- `to_now`: past rain right up to the issue time.
- `perfprog`: perfect prognosis. Observed rain up to `rain_future_hrs` hours *after* the issue time, but never past the lead time. With `rain_future_hrs` = 18: a 6 h lead gets 6 h of future rain, and a 24 h lead gets 18 h.

**Data files**: each file's type is recognized from its columns, not its name:
- `wl, uwind, vwind`: the main station
- `water_level_m`: Port O'Connor
- `precip_in`: hourly IEM rain, resampled to 6 min

So a new location or a renamed file only needs a config change. Each split's files are joined side by side on time, so list one main station file, plus at most one rain file and one POC file.

To add an experiment: copy a config file on GitHub, rename it, edit it, and commit. Then rerun Setup and Pick experiments. Copy instead of overwriting, so old results still match their config. (Every result also stores a copy of its config, so you can always check.)

# Model Training

## Run experiments

1. Runtime → Run all. Sign in to Google if asked. Nothing trains yet.
2. In **Pick experiments**, turn ON the experiments you want (green = on, red = off). You don't need to re-run that cell after clicking.
3. Run **Train selected experiments**.

Each experiment is one config file in the repo's `configs/` folder (see **Config files** near the top). An experiment shows up in red if its config has a problem, with a message saying what to fix.

In [ ]:
#@title Pick experiments (click to turn ON / OFF) { display-mode: "form" }
experiments, bad_configs = load_configs(CONFIG_DIR)
EXPERIMENT_TOGGLES = {}  # experiment name -> (toggle button, experiment)

summary = widgets.HTML()


def update_summary(change=None):
    """Colors each toggle and counts how many models will be trained"""
    n_experiments = 0
    n_models = 0
    for toggle, experiment in EXPERIMENT_TOGGLES.values():
        if toggle.value:
            toggle.button_style = 'success'
            toggle.icon = 'check'
            n_experiments += 1
            n_models += len(experiment['leads']) * experiment['leads'][0]['n_models']
        else:
            toggle.button_style = 'danger'
            toggle.icon = 'times'
    summary.value = f"<b>{n_experiments} experiment(s) selected = {n_models} models to train</b>"


def set_all(value):
    for toggle, _ in EXPERIMENT_TOGGLES.values():
        toggle.value = value


rows = []
for name, experiment in experiments.items():
    cfg = experiment['leads'][0]
    leads = ', '.join(str(c['lead_hrs']) for c in experiment['leads'])
    text = (f"{experiment['location']} | leads {leads} h | rain {cfg['rain']} | POC {cfg['poc']} | "
            f"{cfg['n_models']} models per lead")
    toggle = widgets.ToggleButton(value=False, description=name, layout=widgets.Layout(width='260px'))
    toggle.observe(update_summary, names='value')
    EXPERIMENT_TOGGLES[name] = (toggle, experiment)
    rows.append(widgets.HBox([toggle, widgets.HTML(f"<span style='color:gray'>&nbsp;{html_lib.escape(text)}</span>")]))

all_on = widgets.Button(description='All on', layout=widgets.Layout(width='80px'))
all_off = widgets.Button(description='All off', layout=widgets.Layout(width='80px'))
all_on.on_click(lambda _: set_all(True))
all_off.on_click(lambda _: set_all(False))

panel = [widgets.HBox([all_on, all_off])] + rows
for file_name, error in bad_configs.items():
    panel.append(widgets.HTML(f"<b style='color:red'>✖ {html_lib.escape(file_name)} can't be run: "
                              f"{html_lib.escape(error)}</b>"))
if not experiments and not bad_configs:
    panel.append(widgets.HTML(f"<b style='color:red'>No config files found in {CONFIG_DIR}</b>"))

update_summary()
display(widgets.VBox(panel + [summary]))

### What training saves

Everything goes in the Drive folder under `sessions/<date_time>/`:

- **`<session>_report.html`**: the final report, with every experiment, **validation set only**. This is the one to share. It's made at the very end.
- **`<session>_report_VAL+TEST.html`**: the same report plus the **testing** results, with a big warning at the top. Only send it on request.
- **`<session>_<experiment>_report.html`**: one experiment's report, made as soon as that experiment finishes. It's built in the background while the next experiment trains.
- `checkpoints/`: a small page per finished model. It's deleted automatically once the final report is built, so if it's still there, something went wrong. Those pages still show what finished.
- `<experiment>/<lead>h/m<k>/`: each model's files: model, config, results, predictions, loss history and plots.
- `configs_used/`: a copy of each config file as it was when trained.

Backups while training (made in the background, so training doesn't wait):
- After **every lead time**: a `partial_<experiment>_<lead>h_...` zip of that lead's models.
- After **every experiment**: that experiment's report.
- At the **end**: the final report and a `FINAL_...` zip of the whole session.

Each of these is saved in the Colab session, under `/content/wl_saves/<session>/` (Files panel on the left), and in Drive, under `saves/<session>/`. The checkpoint pages go to both places too.

Only the **two final reports** and the **FINAL zip** are also downloaded to your computer, at the very end. Colab holds downloads back while a cell is running, so mid-run downloads don't work. The browser may ask to allow multiple downloads: click **Allow**.

Also:
- `all_results.csv` (one row per model, every session, validation results) and `failed_runs.csv` in the Drive folder. They include each model's training time.
- Testing results are kept apart, in `all_test_results.csv` and in each model's `_test_results.csv` / `_test_predictions.csv`. They're never printed during training.
- The training cell prints whether a GPU is in use, and how long each model, lead and experiment took.
- If you saw a red "Drive did not connect" warning, the Colab session and the final downloads are the only copies.


In [ ]:
#@title Train selected experiments { display-mode: "form" }
# Everything (inputs, data files, model settings) comes from the config files. Nothing to set here.
import tensorflow as tf

selected = []
for toggle, experiment in EXPERIMENT_TOGGLES.values():
    if toggle.value:
        selected.append(experiment)

if not selected:
    print("No experiments selected. Turn experiments ON in the 'Pick experiments' cell above, then run this cell again.")
else:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"GPU: yes ({len(gpus)})")
    else:
        display(HTML("<b style='color:orange'>GPU: no. Training runs on the CPU and will be much slower. "
                     "In Colab: Runtime → Change runtime type → GPU.</b>"))

    # ---- Pre-flight: load every experiment's data and check its columns, before anything trains ----
    print("Checking the data for every selected experiment...")
    for experiment in selected:
        check_experiment_data(experiment)
    print("OK\n")

    session_id = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    session_dir = Path(OUT_DIR) / "sessions" / session_id
    os.makedirs(session_dir / "configs_used", exist_ok=True)
    if not DRIVE_OK:
        display(HTML("<b style='color:red'>⚠ Drive not connected: the Colab session and the downloads are the only copies.</b>"))

    session_start = time.perf_counter()
    session_results = []
    timings = []  # lead, experiment and session times -> session_timings.csv (in the session folder and the FINAL zip)
    timings_path = session_dir / "session_timings.csv"
    for experiment in selected:
        print(f"==================== {experiment['name']} ====================")
        experiment_start = time.perf_counter()
        shutil.copy(Path(CONFIG_DIR) / experiment['file'], session_dir / "configs_used" / experiment['file'])

        train_df = load_split(experiment['train_files'])
        val_df = load_split(experiment['val_files'])
        test_df = load_split(experiment['test_files'])
        data_fingerprints = {}
        for file_name in experiment['train_files'] + experiment['val_files'] + experiment['test_files']:
            data_fingerprints[file_name] = file_fingerprint(Path(DATA_DIR) / file_name)

        for cfg in experiment['leads']:
            lead_start = time.perf_counter()
            # The inputs are the same for every model of this lead, so build them once
            train = build_arrays(train_df, cfg, cfg['train_gap_fill'])
            val = build_arrays(val_df, cfg, cfg['val_gap_fill'])
            test = build_arrays(test_df, cfg, cfg['test_gap_fill'])  # testing results only go in the VAL+TEST report
            print(f"{cfg['lead_hrs']} h lead: {len(train['input_array']):,} training rows | "
                  f"{len(val['input_array']):,} validation rows | {len(test['target_array']):,} testing rows | "
                  f"{train['input_array'].shape[1]:,} inputs")
            for model_num in range(1, cfg['n_models'] + 1):
                row = train_one_model(experiment, cfg, model_num, train, val, test, session_id, session_dir,
                                      data_fingerprints)
                session_results.append(row)
            del train, val, test  # free memory before the next lead
            lead_minutes = (time.perf_counter() - lead_start) / 60
            print(f"{cfg['lead_hrs']} h lead done in {lead_minutes:.1f} min\n")
            timings.append({'level': 'lead', 'experiment': experiment['name'], 'lead_hrs': cfg['lead_hrs'],
                            'models': cfg['n_models'], 'minutes': round(lead_minutes, 2)})
            pd.DataFrame(timings).to_csv(timings_path, index=False)  # saved after every lead, in case of a crash

            # Backup after every lead: zip this lead's folder (in the background)
            lead_dir = session_dir / experiment['name'] / f"{cfg['lead_hrs']}h"
            zip_name = f"partial_{experiment['name']}_{cfg['lead_hrs']}h_{session_id}"
            save_in_background(zip_name, zip_and_save, lead_dir, zip_name, session_id)

        # After every experiment: its own report (in the background, while the next experiment trains)
        experiment_minutes = (time.perf_counter() - experiment_start) / 60
        print(f"{experiment['name']} done in {experiment_minutes:.1f} min\n")
        timings.append({'level': 'experiment', 'experiment': experiment['name'],
                        'models': len(experiment['leads']) * experiment['leads'][0]['n_models'],
                        'minutes': round(experiment_minutes, 2)})
        pd.DataFrame(timings).to_csv(timings_path, index=False)
        save_in_background(f"{experiment['name']} report", report_and_save, session_dir, experiment['name'], session_id)

    session_df = pd.DataFrame(session_results)
    session_df.to_csv(session_dir / "session_results.csv", index=False)

    print("Waiting for the background saves to finish...")
    wait_for_saves()

    # ---- The final report with everything. Once it exists, the checkpoint pages aren't needed any more ----
    try:
        print("\nBuilding the final report...")
        report_path = build_session_report(session_dir)                        # validation only: the one to share
        test_report_path = build_session_report(session_dir, include_test=True)  # validation + testing: on request
        if report_path.exists() and report_path.stat().st_size > 0:
            shutil.rmtree(session_dir / "checkpoints", ignore_errors=True)
            shutil.rmtree(LOCAL_SAVE_DIR / session_id / "checkpoints", ignore_errors=True)
        save_everywhere(report_path, session_id, download=True)
        save_everywhere(test_report_path, session_id, download=True)
        print(f"Report (validation only, the one to share): {report_path}")
        print(f"Report with TESTING results (only on request): {test_report_path}")
    except Exception as e:
        traceback.print_exc()
        print(f"⚠ The final report could not be built ({e}). The checkpoints folder was kept. "
              f"You can build the report later with the 'Build the report' cells at the bottom.")
    # Copies of the running logs (every session so far) go in the FINAL zip too.
    # Named *_snapshot.csv so the report never mistakes them for a model's _results.csv
    for log_name in ['all_results', 'all_test_results', 'failed_runs']:
        log_path = Path(OUT_DIR) / f"{log_name}.csv"
        if log_path.exists():
            shutil.copy(log_path, session_dir / f"{log_name}_snapshot.csv")

    # Total time goes in the timings file before the FINAL zip is made, so the zip has it too
    total_minutes = (time.perf_counter() - session_start) / 60
    timings.append({'level': 'session (training + reports)', 'models': len(session_results),
                    'minutes': round(total_minutes, 2)})
    pd.DataFrame(timings).to_csv(timings_path, index=False)
    zip_and_save(session_dir, f"FINAL_{session_id}", session_id, download=True)

    print("\n==================== Done ====================")
    print(f"Total time: {total_minutes:.1f} min (timings saved in {timings_path.name})")
    print(f"Saved to: {session_dir}")
    print(f"Copies of the zips and reports: {LOCAL_SAVE_DIR / session_id} (this Colab session)")
    if DRIVE_OK:
        print(f"and {Path(OUT_DIR) / 'saves' / session_id} (Drive)")
        print(f"Master results CSV updated: {OUT_DIR}/all_results.csv")
    else:
        print("Drive was NOT connected. Download the files from the Colab Files panel before closing!")
    if 'rmse' in session_df.columns:
        display(session_df[['run_name', 'status', 'rmse', 'cf_15cm', 'epochs_trained', 'train_seconds']])
    else:
        display(session_df[['run_name', 'status']])


# Load a trained model and evaluate it on the validation set

1. Run **Pick a saved model** and choose a model from the list (newest first). Or paste the full path to a `_model.keras` file.
2. Run **Load and evaluate**. It loads the model and its `_config.json`, rebuilds the validation inputs exactly as in training, and prints the metrics. The testing set is never used.
3. Everything is stored in `run` for your own visuals: `run['y_val']`, `run['predictions']`, `run['datetime_index']`, `run['x_val']`, `run['input_columns']`, `run['metrics']`, `run['cfg']`, `run['model']`.

In [ ]:
#@title Pick a saved model { display-mode: "form" }
model_files = sorted(Path(OUT_DIR).glob("sessions/*/*/*/*/*_model.keras"), key=lambda p: p.stat().st_mtime, reverse=True)
options = []
for p in model_files:
    session_name = p.parents[3].name
    options.append((f"{session_name} / {p.name[:-len('_model.keras')]}", str(p)))
model_picker = widgets.Dropdown(options=options, description='Model:', layout=widgets.Layout(width='700px'))
model_path_box = widgets.Text(placeholder='...or paste the full path to a _model.keras file (overrides the list)',
                              layout=widgets.Layout(width='700px'))
if not model_files:
    print(f"No saved models found in {OUT_DIR}/sessions. Paste a path below instead.")
display(widgets.VBox([model_picker, model_path_box]))

In [ ]:
#@title Load and evaluate (validation set) { display-mode: "form" }
chosen_path = model_path_box.value.strip()
if chosen_path == '':
    chosen_path = model_picker.value
if not chosen_path:
    raise ValueError("No model selected. Pick one in the list above (run that cell first) or paste a path.")
run = load_run(chosen_path)
model = run['model']

In [ ]:
# Example extra visual: predicted vs observed scatter for the loaded run (add your own below)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(run['y_val'], run['predictions'], s=2, alpha=0.3)
lims = [min(run['y_val'].min(), run['predictions'].min()), max(run['y_val'].max(), run['predictions'].max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Observed (m, NAVD88)')
ax.set_ylabel('Predicted (m, NAVD88)')
ax.set_title(f"{run['run_name']}: validation set (RMSE {run['metrics']['rmse']:.3f} m)")
ax.legend()
plt.show()

# Build the report again

The report is built automatically at the end of training. Use these cells only if that failed, or to rebuild a session's report.

Pick a session folder and run **Build the report**. It writes `<session>_report.html` into that folder and deletes its `checkpoints/` folder.

In [ ]:
#@title Pick a session folder { display-mode: "form" }
session_dirs = sorted([p for p in Path(OUT_DIR).glob("sessions/*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
options = []
for p in session_dirs:
    options.append((f"sessions/{p.name}", str(p)))
folder_picker = widgets.Dropdown(options=options, description='Folder:', layout=widgets.Layout(width='700px'))
folder_path_box = widgets.Text(placeholder='...or paste the full path to a session folder (overrides the list)',
                               layout=widgets.Layout(width='700px'))
if not session_dirs:
    print("No session folders found. Paste a path below instead.")
display(widgets.VBox([folder_picker, folder_path_box]))

In [ ]:
#@title Build the report { display-mode: "form" }
DOWNLOAD_REPORT = True  #@param {type:"boolean"}
chosen_folder = folder_path_box.value.strip()
if chosen_folder == '':
    chosen_folder = folder_picker.value
if not chosen_folder:
    raise ValueError("No folder selected. Pick one in the list above (run that cell first) or paste a path.")
report_path = build_session_report(chosen_folder)
test_report_path = build_session_report(chosen_folder, include_test=True)
shutil.rmtree(Path(chosen_folder) / "checkpoints", ignore_errors=True)
shutil.rmtree(LOCAL_SAVE_DIR / Path(chosen_folder).name / "checkpoints", ignore_errors=True)
print(f"Saved: {report_path}  (validation only, the one to share)")
print(f"Saved: {test_report_path}  (with TESTING results, only on request)")
if DOWNLOAD_REPORT:
    try_download(str(report_path))
    try_download(str(test_report_path))

most pressing matters at the moment are:
*    look into how the rain datasets assign the hourly value (is 1pm the rainfall from 12-1 or 1-2?
*    whatever we discover will provide insight into how rain data should be fed nto the model. the 22 hr rain data is ... interesting
*    juliet will look into this